# Ablation Study (Thesis Section 4.4.2)

Analyzes 13 fine-tuning configurations evaluated on the **validation set** (510 samples, 17 questions) to justify hyperparameter selection.

**Sections:**
1. Load All Configuration Results
2. LoRA Rank Sweep (r=8 to 128)
3. Learning Rate Sweep (1e-6 to 2e-5)
4. Loss Curves
5. Extended Training (r=32, 5 epochs)
6. Dropout Analysis (r=80, dropout=0.1)
7. Bootstrap Confidence Intervals (10,000 resamples)
8. Pairwise Significance Tests vs Baseline
9. Final Configuration Selection

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import cohen_kappa_score
from pathlib import Path

sns.set_theme(style='whitegrid', font_scale=1.1)
np.random.seed(42)

RESULTS_DIR = Path('../Ablation Study')

CONFIG_ORDER = [
    'r8', 'r16', 'r32', 'r64', 'r80', 'r96', 'r128',
    'lr1e-06', 'lr5e-06', 'lr1e-05', 'lr2e-05',
    'r32_5epochs', 'r80_dropout0.1'
]

CONFIG_LABELS = {
    'r8': 'r=8', 'r16': 'r=16', 'r32': 'r=32', 'r64': 'r=64',
    'r80': 'r=80 (Baseline)', 'r96': 'r=96', 'r128': 'r=128',
    'lr1e-06': 'LR=1e-6', 'lr5e-06': 'LR=5e-6',
    'lr1e-05': 'LR=1e-5', 'lr2e-05': 'LR=2e-5',
    'r32_5epochs': 'r=32, 5ep', 'r80_dropout0.1': 'r=80, drop=0.1'
}

## 1. Load All Configuration Results

In [ ]:
# Load summary JSONs and validation results
summaries = {}
val_results = {}

for config in CONFIG_ORDER:
    with open(RESULTS_DIR / config / 'summary.json') as f:
        summaries[config] = json.load(f)
    df = pd.read_csv(RESULTS_DIR / config / 'val_results.csv')
    # Normalize column names
    col_map = {}
    for c in df.columns:
        if c in ('Within_1', 'Within_1.0'): col_map[c] = 'Within_1'
        elif c in ('Within_05', 'Within_0.5'): col_map[c] = 'Within_05'
    df = df.rename(columns=col_map)
    val_results[config] = df
    print(f'  {CONFIG_LABELS[config]:22s}: {len(df)} samples, ±1.0 Acc={summaries[config]["acc_1"]:.2f}%')

print(f'\nLoaded {len(summaries)} configurations (all evaluated on validation set)')

In [ ]:
# Master summary table
rows = []
for config in CONFIG_ORDER:
    s = summaries[config]
    rows.append({
        'Config': CONFIG_LABELS[config],
        'LoRA r': s.get('lora_r', ''),
        'LR': s.get('learning_rate', ''),
        'Epochs': s.get('epochs', ''),
        'Acc \u00b11.0 (%)': round(s['acc_1'], 2),
        'Acc \u00b10.5 (%)': round(s['acc_05'], 2),
        'MAE': round(s['mae'], 4),
        'QWK': round(s['qwk'], 4),
        'OT Acc (%)': round(s['off_topic_acc'], 2),
        'Best Epoch': s.get('best_epoch', '')
    })

master_df = pd.DataFrame(rows)
master_df

## 2. LoRA Rank Sweep

In [ ]:
# Load pre-computed summary
rank_df = pd.read_csv(RESULTS_DIR / 'summary_lora_rank.csv')
print('LoRA Rank Sweep Results:')
rank_df[['config', 'lora_r', 'Acc_1.0', 'Acc_0.5', 'MAE', 'QWK', 'OffTopic_Acc', 'Best_Epoch', 'Best_Eval_Loss']]

In [ ]:
# Annotated LoRA rank plots
rank_configs = ['r8', 'r16', 'r32', 'r64', 'r80', 'r96', 'r128']
ranks = [summaries[c]['lora_r'] for c in rank_configs]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
metrics = [
    ('acc_1', 'Acc \u00b11.0 (%)', True),
    ('acc_05', 'Acc \u00b10.5 (%)', True),
    ('mae', 'MAE', False),
    ('qwk', 'QWK', True)
]

for ax, (key, label, higher_better) in zip(axes.flat, metrics):
    values = [summaries[c][key] for c in rank_configs]
    ax.plot(ranks, values, 'o-', linewidth=2, markersize=8, color='#3498db')

    # Highlight best
    best_idx = np.argmax(values) if higher_better else np.argmin(values)
    ax.plot(ranks[best_idx], values[best_idx], 'o', markersize=14,
            color='#2ecc71', zorder=5, label=f'Best: r={ranks[best_idx]}')

    # Annotate each point
    for i, (r, v) in enumerate(zip(ranks, values)):
        fmt = f'{v:.1f}%' if '%' in label else f'{v:.4f}'
        ax.annotate(fmt, (r, v), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8)

    ax.set_xlabel('LoRA Rank (r)', fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('LoRA Rank Ablation (Validation Set)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 3. Learning Rate Sweep

In [ ]:
lr_df = pd.read_csv(RESULTS_DIR / 'summary_learning_rate.csv')
print('Learning Rate Sweep Results:')
lr_df[['config', 'learning_rate', 'Acc_1.0', 'Acc_0.5', 'MAE', 'QWK', 'OffTopic_Acc', 'Best_Epoch']]

In [ ]:
# Annotated LR plots
lr_configs = ['lr1e-06', 'lr5e-06', 'lr1e-05', 'lr2e-05']
lrs = [summaries[c]['learning_rate'] for c in lr_configs]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))

for ax, (key, label, higher_better) in zip(axes.flat, metrics):
    values = [summaries[c][key] for c in lr_configs]
    ax.plot(range(len(lrs)), values, 'o-', linewidth=2, markersize=8, color='#e74c3c')

    best_idx = np.argmax(values) if higher_better else np.argmin(values)
    ax.plot(best_idx, values[best_idx], 'o', markersize=14,
            color='#2ecc71', zorder=5, label=f'Best: LR={lrs[best_idx]}')

    for i, (lr, v) in enumerate(zip(lrs, values)):
        fmt = f'{v:.1f}%' if '%' in label else f'{v:.4f}'
        ax.annotate(fmt, (i, v), textcoords='offset points',
                    xytext=(0, 10), ha='center', fontsize=8)

    ax.set_xticks(range(len(lrs)))
    ax.set_xticklabels([f'{lr:.0e}' for lr in lrs])
    ax.set_xlabel('Learning Rate', fontsize=11)
    ax.set_ylabel(label, fontsize=11)
    ax.set_title(label, fontsize=13, fontweight='bold')
    ax.legend(fontsize=9)

fig.suptitle('Learning Rate Ablation (r=80, Validation Set)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

## 4. Loss Curves

In [ ]:
rank_configs = ['r8', 'r16', 'r32', 'r64', 'r80', 'r96', 'r128']

# Training loss curves
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for config in rank_configs:
    train_loss = pd.read_csv(RESULTS_DIR / config / 'train_loss.csv')
    ax.plot(train_loss['step'], train_loss['train_loss'],
            label=CONFIG_LABELS[config], alpha=0.8)
ax.set_xlabel('Training Step')
ax.set_ylabel('Training Loss')
ax.set_title('(a) Training Loss \u2014 LoRA Rank Sweep', fontweight='bold')
ax.legend(fontsize=8, ncol=2)

ax = axes[1]
for config in lr_configs:
    train_loss = pd.read_csv(RESULTS_DIR / config / 'train_loss.csv')
    ax.plot(train_loss['step'], train_loss['train_loss'],
            label=CONFIG_LABELS[config], alpha=0.8)
ax.set_xlabel('Training Step')
ax.set_ylabel('Training Loss')
ax.set_title('(b) Training Loss \u2014 Learning Rate Sweep', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluation loss by epoch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

ax = axes[0]
for config in rank_configs:
    eval_loss = pd.read_csv(RESULTS_DIR / config / 'eval_loss.csv')
    ax.plot(eval_loss['epoch'], eval_loss['eval_loss'], 'o-',
            label=CONFIG_LABELS[config], markersize=6)
ax.set_xlabel('Epoch')
ax.set_ylabel('Eval Loss')
ax.set_title('(a) Eval Loss \u2014 LoRA Rank Sweep', fontweight='bold')
ax.legend(fontsize=8, ncol=2)

ax = axes[1]
for config in lr_configs:
    eval_loss = pd.read_csv(RESULTS_DIR / config / 'eval_loss.csv')
    ax.plot(eval_loss['epoch'], eval_loss['eval_loss'], 'o-',
            label=CONFIG_LABELS[config], markersize=6)
ax.set_xlabel('Epoch')
ax.set_ylabel('Eval Loss')
ax.set_title('(b) Eval Loss \u2014 Learning Rate Sweep', fontweight='bold')
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

## 5. Extended Training Analysis (r=32, 5 epochs)

In [ ]:
ext_configs = ['r32', 'r32_5epochs']
ext_labels = ['r=32, 3 epochs', 'r=32, 5 epochs']

ext_rows = []
for config, label in zip(ext_configs, ext_labels):
    s = summaries[config]
    ext_rows.append({
        'Config': label,
        'Acc \u00b11.0 (%)': round(s['acc_1'], 2),
        'Acc \u00b10.5 (%)': round(s['acc_05'], 2),
        'MAE': round(s['mae'], 4),
        'QWK': round(s['qwk'], 4),
        'OT Acc (%)': round(s['off_topic_acc'], 2),
        'Best Epoch': s.get('best_epoch', ''),
        'Best Eval Loss': round(s.get('best_eval_loss', 0), 4)
    })

print('Extended Training Comparison:')
pd.DataFrame(ext_rows)

In [ ]:
# Loss curves: 3 epoch vs 5 epoch
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, loss_type, title in [(axes[0], 'train_loss', 'Training Loss'),
                              (axes[1], 'eval_loss', 'Eval Loss')]:
    for config, label, color in zip(ext_configs, ext_labels, ['#3498db', '#e74c3c']):
        loss_file = f'{loss_type.split("_")[0]}_loss.csv'
        loss_df = pd.read_csv(RESULTS_DIR / config / loss_file)
        col = loss_df.columns[0]
        val_col = loss_df.columns[1]
        ax.plot(loss_df[col], loss_df[val_col], 'o-' if 'eval' in loss_type else '-',
                label=label, color=color, markersize=6)
    ax.set_xlabel(loss_df.columns[0].capitalize())
    ax.set_ylabel(title)
    ax.set_title(title, fontweight='bold')
    ax.legend()

fig.suptitle('Extended Training: 3 vs 5 Epochs (r=32)', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 6. Dropout Analysis

In [ ]:
drop_configs = ['r80', 'r80_dropout0.1']
drop_labels = ['r=80, dropout=0', 'r=80, dropout=0.1']

drop_rows = []
for config, label in zip(drop_configs, drop_labels):
    s = summaries[config]
    drop_rows.append({
        'Config': label,
        'Acc \u00b11.0 (%)': round(s['acc_1'], 2),
        'Acc \u00b10.5 (%)': round(s['acc_05'], 2),
        'MAE': round(s['mae'], 4),
        'QWK': round(s['qwk'], 4),
        'OT Acc (%)': round(s['off_topic_acc'], 2),
    })

print('Dropout Comparison (r=80):')
pd.DataFrame(drop_rows)

## 7. Bootstrap Confidence Intervals (10,000 resamples)

Compute bootstrap CIs directly from the validation results.

In [ ]:
def compute_metrics_from_df(df):
    """Compute metrics from a results DataFrame (vectorized for speed)."""
    human_finals = df['Human_Final'].values
    llm_finals = df['LLM_Final'].values
    diffs = np.abs(human_finals - llm_finals)

    # Total scores for QWK
    human_totals = (df['Human_Clarity'] + df['Human_Terminology'] +
                    df['Human_Coverage'] + df['Human_Accuracy']).values
    llm_totals = (df['LLM_Clarity'] + df['LLM_Terminology'] +
                  df['LLM_Coverage'] + df['LLM_Accuracy']).values

    # Off-topic accuracy (vectorized)
    h_ot = df['Human_OffTopic'].isin(['Off-Topic', 'No Answer']).values
    l_ot = df['LLM_OffTopic'].isin(['Off-Topic', 'No Answer']).values
    ot_correct = np.sum((h_ot & l_ot) | (~h_ot & ~l_ot))

    return {
        'acc_1': np.mean(diffs <= 1.0) * 100,
        'acc_05': np.mean(diffs <= 0.5) * 100,
        'mae': np.mean(diffs),
        'qwk': cohen_kappa_score(human_totals, llm_totals, weights='quadratic'),
        'off_topic_acc': ot_correct / len(df) * 100
    }


def bootstrap_config(df, n_bootstrap=10000):
    """Run bootstrap resampling for a single configuration."""
    n = len(df)
    boot_metrics = {k: [] for k in ['acc_1', 'acc_05', 'mae', 'qwk', 'off_topic_acc']}

    for _ in range(n_bootstrap):
        idx = np.random.randint(0, n, size=n)
        sample = df.iloc[idx]
        m = compute_metrics_from_df(sample)
        for k in boot_metrics:
            boot_metrics[k].append(m[k])

    return {k: np.array(v) for k, v in boot_metrics.items()}


print('Running bootstrap analysis (10,000 resamples x 13 configs)...')
print('This takes ~2-3 minutes.\n')

N_BOOTSTRAP = 10000
bootstrap_results = {}

for i, config in enumerate(CONFIG_ORDER):
    boot = bootstrap_config(val_results[config], N_BOOTSTRAP)
    point = compute_metrics_from_df(val_results[config])

    bootstrap_results[config] = {
        'point': point,
        'boot': boot,
        'ci': {k: (np.percentile(boot[k], 2.5), np.percentile(boot[k], 97.5))
               for k in boot}
    }

    ci = bootstrap_results[config]['ci']
    print(f'  [{i+1:2d}/13] {CONFIG_LABELS[config]:22s}  '
          f'Acc\u00b11: {point["acc_1"]:.1f}% [{ci["acc_1"][0]:.1f}, {ci["acc_1"][1]:.1f}]  '
          f'QWK: {point["qwk"]:.3f} [{ci["qwk"][0]:.3f}, {ci["qwk"][1]:.3f}]')

print('\nBootstrap complete.')

In [ ]:
# Build CI summary table
ci_rows = []
for config in CONFIG_ORDER:
    br = bootstrap_results[config]
    row = {'Configuration': CONFIG_LABELS[config]}
    for metric, label in [('acc_1', 'Acc \u00b11.0 (%)'), ('acc_05', 'Acc \u00b10.5 (%)'),
                          ('mae', 'MAE'), ('qwk', 'QWK'), ('off_topic_acc', 'Off-Topic Acc (%)')]:
        row[f'{label} Point'] = round(br['point'][metric], 3)
        row[f'{label} CI Lower'] = round(br['ci'][metric][0], 3)
        row[f'{label} CI Upper'] = round(br['ci'][metric][1], 3)
    ci_rows.append(row)

ci_df = pd.DataFrame(ci_rows)

# Display key CIs
display_cols = ['Configuration',
    'Acc \u00b11.0 (%) Point', 'Acc \u00b11.0 (%) CI Lower', 'Acc \u00b11.0 (%) CI Upper',
    'MAE Point', 'MAE CI Lower', 'MAE CI Upper',
    'QWK Point', 'QWK CI Lower', 'QWK CI Upper']
ci_df[display_cols]

In [ ]:
# CI forest plot
fig, axes = plt.subplots(2, 2, figsize=(18, 14))
fig.suptitle('Bootstrap 95% Confidence Intervals \u2014 All 13 Configurations (Validation Set)',
             fontsize=16, fontweight='bold')

metrics_plot = [
    ('Acc \u00b11.0 (%)', '{:.1f}%'),
    ('Acc \u00b10.5 (%)', '{:.1f}%'),
    ('MAE', '{:.3f}'),
    ('QWK', '{:.3f}')
]

for ax_idx, (metric, fmt) in enumerate(metrics_plot):
    ax = axes[ax_idx // 2][ax_idx % 2]
    labels = ci_df['Configuration'].values
    points = ci_df[f'{metric} Point'].values
    lowers = points - ci_df[f'{metric} CI Lower'].values
    uppers = ci_df[f'{metric} CI Upper'].values - points

    y_pos = np.arange(len(labels))
    colors = ['#2ecc71' if 'Baseline' in l else
              '#e74c3c' if 'LR' in l else
              '#9b59b6' if '5ep' in l or 'drop' in l else
              '#3498db' for l in labels]

    ax.barh(y_pos, points, xerr=[lowers, uppers], height=0.55,
            color=colors, edgecolor='gray', capsize=3, alpha=0.85)
    ax.set_yticks(y_pos)
    ax.set_yticklabels(labels, fontsize=9)
    ax.set_xlabel(metric, fontsize=11)
    ax.set_title(metric, fontweight='bold', fontsize=13)
    ax.invert_yaxis()

    baseline_idx = list(labels).index('r=80 (Baseline)')
    ax.axvline(points[baseline_idx], color='#2ecc71', linestyle='--', alpha=0.6, linewidth=1.5)

from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#2ecc71', label='Baseline (r=80)'),
    Patch(facecolor='#3498db', label='LoRA Rank'),
    Patch(facecolor='#e74c3c', label='Learning Rate'),
    Patch(facecolor='#9b59b6', label='Extended/Dropout')
]
fig.legend(handles=legend_elements, loc='lower center', ncol=4, fontsize=11,
           bbox_to_anchor=(0.5, -0.02))
plt.tight_layout()
plt.show()

## 8. Pairwise Significance Tests vs Baseline (r=80)

In [ ]:
def pairwise_bootstrap_test(boot_baseline, boot_other, metric, higher_better=True):
    """Two-sided bootstrap test: proportion of times baseline is better."""
    base_vals = boot_baseline[metric]
    other_vals = boot_other[metric]
    if higher_better:
        p_value = np.mean(other_vals >= base_vals)
        direction = 'baseline_better' if np.mean(base_vals) > np.mean(other_vals) else 'other_better'
    else:
        p_value = np.mean(other_vals <= base_vals)
        direction = 'baseline_better' if np.mean(base_vals) < np.mean(other_vals) else 'other_better'

    if p_value < 0.001:
        sig = '***'
    elif p_value < 0.01:
        sig = '**'
    elif p_value < 0.05:
        sig = '*'
    else:
        sig = 'ns'

    return p_value, sig, direction


# Run pairwise tests vs r=80 baseline
baseline_boot = bootstrap_results['r80']['boot']
non_baseline = [c for c in CONFIG_ORDER if c != 'r80']

sig_rows = []
for config in non_baseline:
    other_boot = bootstrap_results[config]['boot']
    row = {'Configuration': CONFIG_LABELS[config]}

    for metric, label, higher_better in [
        ('acc_1', 'Acc \u00b11.0 (%)', True),
        ('acc_05', 'Acc \u00b10.5 (%)', True),
        ('mae', 'MAE', False),
        ('qwk', 'QWK', True),
        ('off_topic_acc', 'Off-Topic Acc (%)', True)
    ]:
        p, sig, direction = pairwise_bootstrap_test(baseline_boot, other_boot, metric, higher_better)
        row[f'{label} p-value'] = round(p, 4)
        row[f'{label} sig'] = sig
        row[f'{label} direction'] = direction

    sig_rows.append(row)

sig_df = pd.DataFrame(sig_rows)

# Display summary
sig_summary = sig_df[['Configuration']].copy()
for metric in ['Acc \u00b11.0 (%)', 'Acc \u00b10.5 (%)', 'MAE', 'QWK', 'Off-Topic Acc (%)']:
    p_col = f'{metric} p-value'
    sig_col = f'{metric} sig'
    sig_summary[metric] = sig_df[p_col].apply(lambda p: f'{p:.3f}') + ' ' + sig_df[sig_col]

print('Pairwise Bootstrap Significance vs r=80 Baseline:')
sig_summary

In [ ]:
# Significance heatmap
metrics_hm = ['Acc \u00b11.0 (%)', 'Acc \u00b10.5 (%)', 'MAE', 'QWK', 'Off-Topic Acc (%)']
configs_hm = sig_df['Configuration'].values

p_matrix = np.zeros((len(configs_hm), len(metrics_hm)))
annot_matrix = []

for i, config in enumerate(configs_hm):
    row_annot = []
    for j, metric in enumerate(metrics_hm):
        p_val = sig_df.iloc[i][f'{metric} p-value']
        sig = sig_df.iloc[i][f'{metric} sig']
        p_matrix[i, j] = p_val
        row_annot.append(f'p={p_val:.3f}\n{sig}')
    annot_matrix.append(row_annot)

annot_matrix = np.array(annot_matrix)

fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(p_matrix, annot=annot_matrix, fmt='',
            xticklabels=metrics_hm, yticklabels=configs_hm,
            cmap='RdYlGn', vmin=0, vmax=1,
            linewidths=0.5, ax=ax,
            cbar_kws={'label': 'p-value (lower = baseline significantly better)'})
ax.set_title('Pairwise Significance vs Baseline (r=80)\nBootstrap test, 10,000 iterations (Validation Set)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Final Configuration Selection

In [ ]:
# Find best config by LR sweep (since r=80 rank was selected)
best_lr_config = max(lr_configs, key=lambda c: summaries[c]['acc_1'])
best = summaries[best_lr_config]

print('=' * 60)
print('FINAL CONFIGURATION SELECTION SUMMARY')
print('=' * 60)

print(f'\nSelected Configuration: r={best["lora_r"]}, LR={best["learning_rate"]}, '
      f'{best["epochs"]} epochs, dropout={best["lora_dropout"]}')
print(f'\nValidation Set Performance:')
print(f'  Acc \u00b11.0: {best["acc_1"]:.2f}%')
print(f'  Acc \u00b10.5: {best["acc_05"]:.2f}%')
print(f'  MAE:      {best["mae"]:.4f}')
print(f'  QWK:      {best["qwk"]:.4f}')
print(f'  OT Acc:   {best["off_topic_acc"]:.2f}%')

print('\nJustification:')
print('  1. LoRA Rank: r=80 achieves peak \u00b11.0 accuracy among rank sweep')
print('  2. Learning Rate: Best LR selected by validation performance')
print('  3. Extended Training: 5 epochs shows diminishing returns / overfitting')
print('  4. Dropout: dropout=0.1 provides marginal difference (unnecessary regularization)')
print('  5. Bootstrap tests confirm statistical significance of configuration choices')